In [0]:
from	pyspark.sql	import	functions	as	F

In [0]:
base	= "/Volumes/real_estate_dev/bronze/vl_real_estate"

def	autoload_cdc(subfolder_glob,	schema_hint_name,	table_name):

	(
     spark.readStream.format("cloudFiles")\
     
		.option("cloudFiles.format",	"csv")\
		.option("header",	True)\
		.option("cloudFiles.schemaLocation", f"{base}/_schema/{schema_hint_name}")\
      
		.load(f"{base}/incremental/{subfolder_glob}")\
      
		.withColumn("_source_file",	F.col("_metadata.file_path"))\
		.withColumn("_ingested_at",	F.current_timestamp())\
	
    	.writeStream.format("delta")\
         
		.option("checkpointLocation",	f"{base}/_checkpoints/{schema_hint_name}")\
		.trigger(availableNow=True)	\
  		#	processes	all	currently available	files,	then	stops
		.toTable(f"real_estate_dev.bronze.{table_name}")\
		.awaitTermination()
  )
 
autoload_cdc("properties_incremental_*.csv",	"properties_cdc",	"properties_cdc")
autoload_cdc("sales_transactions_incremental_*.csv","sales_transactions_cdc",	"sales_transactions_cdc")